# Backtest Data Fetch — February 2026

Fetches real Polymarket touch market probabilities and BTC spot price for the **start of February 2026**.
Outputs `upside_touch`, `downside_touch`, and `SPOT` in the exact format `skew_model.ipynb` expects.

In [44]:
import re
import json
import requests

GAMMA = "https://gamma-api.polymarket.com"
CLOB  = "https://clob.polymarket.com"

MONTH_LABEL    = "February 2026"
MONTH_START_TS = 1769904000   # Feb 1 2026 00:00:00 UTC
MONTH_END_TS   = 1769904000 + 6 * 3600  # first 6 hours — wide window to catch first trade

## Step 1 — Fetch the February 2026 BTC price event

**API:** `GET https://gamma-api.polymarket.com/events/slug/{slug}`  
Docs: https://docs.polymarket.com (Gamma API > Get event by slug)

The Gamma API **does not support text search** — there is no `q=` parameter. Instead, monthly BTC price events follow a predictable slug pattern:

```
what-price-will-bitcoin-hit-in-{month}-{year}
```

Each event bundles ~27 child markets covering both upside ("Will Bitcoin reach $X?") and downside ("Will Bitcoin dip to $X?") targets for the month. This was confirmed by fetching the slug directly and inspecting the `markets[]` array.

In [45]:
# Slug pattern: what-price-will-bitcoin-hit-in-{month}-{year}
EVENT_SLUG = "what-price-will-bitcoin-hit-in-february-2026"

resp = requests.get(f"{GAMMA}/events/slug/{EVENT_SLUG}")
resp.raise_for_status()
event = resp.json()

markets = event.get("markets", [])
print(f"Event: {event['title']}")
print(f"Status: {'closed' if event.get('closed') else 'active'}")
print(f"End date: {event.get('endDate')}")
print(f"Markets: {len(markets)}")
print()
for m in markets:
    print(f"  {m['question']}")

Event: What price will Bitcoin hit in February?
Status: closed
End date: 2026-03-01T05:00:00Z
Markets: 23

  Will Bitcoin reach $150,000 in February?
  Will Bitcoin reach $120,000 in February?
  Will Bitcoin reach $125,000 in February?
  Will Bitcoin dip to $60,000 in February?
  Will Bitcoin reach $115,000 in February?
  Will Bitcoin dip to $55,000 in February?
  Will Bitcoin dip to $70,000 in February?
  Will Bitcoin reach $110,000 in February?
  Will Bitcoin dip to $50,000 in February?
  Will Bitcoin reach $105,000 in February?
  Will Bitcoin reach $100,000 in February?
  Will Bitcoin dip to $80,000 in February?
  Will Bitcoin reach $95,000 in February?
  Will Bitcoin reach $90,000 in February?
  Will Bitcoin dip to $40,000 in February?
  Will Bitcoin reach $85,000 in February?
  Will Bitcoin dip to $75,000 in February?
  Will Bitcoin dip to $65,000 in February?
  Will Bitcoin dip to $45,000 in February?
  Will Bitcoin dip to $35,000 in February?
  Will Bitcoin reach $75,000 in Febr

## Step 2 — Parse markets and classify upside / downside

**Market question patterns observed:**
- Upside: `"Will Bitcoin reach $X,000 in February?"`
- Downside: `"Will Bitcoin dip to $X,000 in February?"`

Some markets have a date qualifier (e.g. "in February (after Feb 6)?"). These are excluded — they measure a conditional touch (only valid after a specific date), which is a different question than the unconditional monthly touch our model uses.

**Token ID convention:** Each market has `clobTokenIds: [YES_token, NO_token]`. We use `clobTokenIds[0]` (the YES outcome token). In a binary prediction market, `price(YES token) = implied probability`.

In [46]:
def parse_strike(question: str) -> int | None:
    """Extract dollar strike price from a question string."""
    m = re.search(r'\$([0-9][0-9,]*)([kK])?', question)
    if not m:
        return None
    digits = int(m.group(1).replace(',', ''))
    if m.group(2):  # k/K suffix
        digits *= 1000
    return digits

def has_date_qualifier(question: str) -> bool:
    """Exclude markets with conditional date qualifiers like '(after Feb 6)'."""
    return bool(re.search(r'after|before|starting|from\s+feb', question, re.IGNORECASE))

upside_markets   = []  # {strike, token_id, question}
downside_markets = []
skipped          = []

for mkt in markets:
    question = mkt.get("question", "")

    if has_date_qualifier(question):
        skipped.append((question, "date-qualified variant"))
        continue

    token_ids = mkt.get("clobTokenIds", [])
    if isinstance(token_ids, str):
        token_ids = json.loads(token_ids)
    if not token_ids:
        skipped.append((question, "no token IDs"))
        continue

    strike = parse_strike(question)
    if strike is None:
        skipped.append((question, "could not parse strike"))
        continue

    q_lower = question.lower()
    if "reach" in q_lower:
        upside_markets.append({"strike": strike, "token_id": token_ids[0], "question": question})
    elif "dip" in q_lower:
        downside_markets.append({"strike": strike, "token_id": token_ids[0], "question": question})
    else:
        skipped.append((question, "unrecognized direction"))

upside_markets.sort(key=lambda x: x["strike"])
downside_markets.sort(key=lambda x: x["strike"], reverse=True)

print("Upside markets (reach $X):")
for m in upside_markets:
    print(f"  ${m['strike']:,}")

print("\nDownside markets (dip to $X):")
for m in downside_markets:
    print(f"  ${m['strike']:,}")

if skipped:
    print(f"\nSkipped ({len(skipped)}):")
    for q, reason in skipped:
        print(f"  [{reason}] {q}")

Upside markets (reach $X):
  $75,000
  $80,000
  $85,000
  $90,000
  $95,000
  $100,000
  $105,000
  $110,000
  $115,000
  $120,000
  $125,000
  $150,000

Downside markets (dip to $X):
  $80,000
  $75,000
  $70,000
  $65,000
  $60,000
  $60,000
  $55,000
  $50,000
  $45,000
  $40,000
  $35,000


## Step 3 — Fetch probabilities at start of February 2026

**API:** `GET https://clob.polymarket.com/prices-history`  
Docs: https://docs.polymarket.com (CLOB API > Get prices history)

**Parameters used:**
| Parameter | Value | Purpose |
|---|---|---|
| `market` | token ID | The YES outcome token for each market |
| `startTs` | `1738368000` | Feb 1 2026 00:00:00 UTC |
| `endTs` | `1738368000 + 21600` | +6 hours — wide enough to catch the first trade |
| `fidelity` | `1` | 1-minute resolution (most granular) |

The response is `{history: [{t: unix_timestamp, p: price}, ...]}`. We take the **first data point** in the window. For a binary market, `p` is the implied probability that the YES outcome resolves.

**Note on resolved market prices:** By the time this data is fetched, these markets have resolved (Feb ended). The `outcomePrices` field on the market object shows the final resolution value (0 or 1), not the probability mid-month. The `/prices-history` endpoint correctly returns the historical time-series.

In [47]:
def fetch_prob_at_month_start(token_id: str) -> tuple[float | None, str]:
    """
    Return (probability, reason) for the YES token at month start.
    probability is None if no data was found in the opening window.
    reason explains why — either the price or why it's absent.
    """
    r = requests.get(f"{CLOB}/prices-history", params={
        "market":   token_id,
        "startTs":  MONTH_START_TS,
        "endTs":    MONTH_END_TS,
        "fidelity": 1,
    })
    r.raise_for_status()
    history = r.json().get("history", [])

    if history:
        return round(float(history[0]["p"]), 4), "ok"

    # No data in opening window — check if the market has any data at all
    # to distinguish "created mid-month" from a genuine API/token problem
    r2 = requests.get(f"{CLOB}/prices-history", params={
        "market":   token_id,
        "interval": "all",
        "fidelity": 1440,
    })
    r2.raise_for_status()
    all_history = r2.json().get("history", [])

    if not all_history:
        return None, "no data at any point — token ID may be invalid"

    first_ts = all_history[0]["t"]
    days_after_start = (first_ts - MONTH_START_TS) / 86400
    return None, f"created mid-month (~day {days_after_start:.0f} of February)"

upside_touch   = {}
downside_touch = {}

print("Fetching upside probabilities...")
for m in upside_markets:
    prob, reason = fetch_prob_at_month_start(m["token_id"])
    if prob is not None:
        upside_touch[m["strike"]] = prob
        print(f"  ${m['strike']:,} -> {prob:.2%}")
    else:
        print(f"  ${m['strike']:,} -> skipped ({reason})")

print("\nFetching downside probabilities...")
for m in downside_markets:
    prob, reason = fetch_prob_at_month_start(m["token_id"])
    if prob is not None:
        downside_touch[m["strike"]] = prob
        print(f"  ${m['strike']:,} -> {prob:.2%}")
    else:
        print(f"  ${m['strike']:,} -> skipped ({reason})")

Fetching upside probabilities...
  $75,000 -> skipped (created mid-month (~day 6 of February))
  $80,000 -> skipped (created mid-month (~day 6 of February))
  $85,000 -> 55.00%
  $90,000 -> 27.00%
  $95,000 -> 15.00%
  $100,000 -> 8.00%
  $105,000 -> 5.30%
  $110,000 -> 3.55%
  $115,000 -> 2.50%
  $120,000 -> 2.70%
  $125,000 -> 2.15%
  $150,000 -> 1.15%

Fetching downside probabilities...
  $80,000 -> 98.60%
  $75,000 -> 67.50%
  $70,000 -> 34.00%
  $65,000 -> 16.00%
  $60,000 -> 9.00%
  $60,000 -> skipped (created mid-month (~day 6 of February))
  $55,000 -> 5.80%
  $50,000 -> 3.45%
  $45,000 -> 2.85%
  $40,000 -> 2.30%
  $35,000 -> 2.10%


## Step 4 — Fetch BTC spot price at February 1, 2026

**API:** `GET https://api.coingecko.com/api/v3/coins/bitcoin/history`  
Docs: https://docs.coingecko.com/reference/coins-id-history

**Parameters:**
| Parameter | Value |
|---|---|
| `date` | `01-02-2026` (DD-MM-YYYY) |
| `localization` | `false` |

Returns daily market data including `market_data.current_price.usd`. No API key required for the free tier.

**Why not use Polymarket for spot?** Polymarket is a prediction market — it does not publish a BTC spot price feed. Spot must come from an external source.

In [48]:
from datetime import datetime, timezone

dt = datetime.fromtimestamp(MONTH_START_TS, tz=timezone.utc)
cg_date = dt.strftime("%d-%m-%Y")  # CoinGecko expects DD-MM-YYYY

r = requests.get(
    "https://api.coingecko.com/api/v3/coins/bitcoin/history",
    params={"date": cg_date, "localization": "false"},
    headers={"accept": "application/json"},
)
r.raise_for_status()
data = r.json()
SPOT = round(data["market_data"]["current_price"]["usd"])
print(f"BTC spot on {dt.strftime('%B %-d, %Y')}: ${SPOT:,}  (CoinGecko date: {cg_date})")

BTC spot on February 1, 2026: $78,726  (CoinGecko date: 01-02-2026)


## Step 5 — Sanity checks

In [49]:
# --- Remove cross-side entries ---
# Markets where the strike is on the wrong side of SPOT are either already
# resolved (e.g. "dip to $80K" when SPOT is $78K — BTC already crossed that
# level) or measure the opposite direction. Remove them before checking.

cross_side = []

for strike in list(upside_touch):
    if strike <= SPOT:
        cross_side.append((strike, "upside", f"strike \\${strike:,} ≤ SPOT \\${SPOT:,} — already crossed, not a true upside target"))
        del upside_touch[strike]

for strike in list(downside_touch):
    if strike >= SPOT:
        cross_side.append((strike, "downside", f"strike \\${strike:,} ≥ SPOT \\${SPOT:,} — already crossed, not a true downside target"))
        del downside_touch[strike]

if cross_side:
    print("Removed cross-side entries:")
    for strike, side, reason in cross_side:
        print(f"  [{side}] \\${strike:,}: {reason}")
    print()

# --- Sanity checks ---
errors = []

for strike in upside_touch:
    if strike <= SPOT:
        errors.append(f"Upside strike \\${strike:,} is not above SPOT \\${SPOT:,}")

for strike in downside_touch:
    if strike >= SPOT:
        errors.append(f"Downside strike \\${strike:,} is not below SPOT \\${SPOT:,}")

for label, d in [("upside", upside_touch), ("downside", downside_touch)]:
    for strike, prob in d.items():
        if not (0 < prob < 1):
            errors.append(f"{label} \\${strike:,} prob {prob} not in (0, 1)")

up_sorted = sorted(upside_touch.items())
for (s1, p1), (s2, p2) in zip(up_sorted, up_sorted[1:]):
    if p2 > p1:
        errors.append(f"Upside non-monotone: \\${s1:,}={p1:.2%} then \\${s2:,}={p2:.2%}")

dn_sorted = sorted(downside_touch.items(), reverse=True)
for (s1, p1), (s2, p2) in zip(dn_sorted, dn_sorted[1:]):
    if p2 > p1:
        errors.append(f"Downside non-monotone: \\${s1:,}={p1:.2%} then \\${s2:,}={p2:.2%}")

if errors:
    print("WARNINGS:")
    for e in errors:
        print(f"  {e}")
else:
    print("All checks passed.")

Removed cross-side entries:
  [downside] \$80,000: strike \$80,000 ≥ SPOT \$78,726 — already crossed, not a true downside target

WARNINGS:
  Upside non-monotone: \$115,000=2.50% then \$120,000=2.70%


## Step 6 — Final assembled inputs

Copy-paste these into `skew_model.ipynb` cell 2 to replace the hardcoded sample data.

In [50]:
print(f"SPOT = {SPOT}")
print()
print("upside_touch = {")
for strike, prob in sorted(upside_touch.items()):
    print(f"    {strike:_}: {prob},")
print("}")
print()
print("downside_touch = {")
for strike, prob in sorted(downside_touch.items(), reverse=True):
    print(f"    {strike:_}: {prob},")
print("}")

SPOT = 78726

upside_touch = {
    85_000: 0.55,
    90_000: 0.27,
    95_000: 0.15,
    100_000: 0.08,
    105_000: 0.053,
    110_000: 0.0355,
    115_000: 0.025,
    120_000: 0.027,
    125_000: 0.0215,
    150_000: 0.0115,
}

downside_touch = {
    75_000: 0.675,
    70_000: 0.34,
    65_000: 0.16,
    60_000: 0.09,
    55_000: 0.058,
    50_000: 0.0345,
    45_000: 0.0285,
    40_000: 0.023,
    35_000: 0.021,
}
